# Task F — Finance question: Credit-card fraud detection

This notebook answers the task as a **design-and-reasoning report** with visible evidence.

## Problem
Perform **binary fraud detection** using the Credit Card Fraud Detection dataset.

## Required evidence included
- EDA first
- one split table (time-aware)
- one class-imbalance handling note
- one lighter baseline and one stronger model
- one threshold-cost table with at least five thresholds
- one confusion matrix
- one short deployment reflection

## Core design choices
- **Target**: `Class` where `1 = fraud` and `0 = legitimate`
- **Evaluation**: **time-aware** split instead of a fully random split
- **Decision rule**: cost-sensitive thresholding because missing fraud is usually much more expensive than raising a manual review

## Task / Question
**Import the required libraries, define a clean visualization style, and configure pandas so the report tables are easy to read.**

In [ ]:

import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (9, 5)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 120)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## Task / Question
**Load the Credit Card Fraud Detection dataset and perform a first sanity check.**

> Expected file name: `creditcard.csv`  
> Put the CSV in the same folder as the notebook, or update `DATA_PATH` below.

In [ ]:

candidate_paths = [
    Path("creditcard.csv"),
    Path("/mnt/data/creditcard.csv")
]

for path in candidate_paths:
    if path.exists():
        DATA_PATH = path
        break
else:
    raise FileNotFoundError(
        "Could not find 'creditcard.csv'. Place it next to the notebook or update DATA_PATH."
    )

df = pd.read_csv(DATA_PATH)

overview = pd.DataFrame({
    "Property": [
        "Rows",
        "Columns",
        "Duplicate rows",
        "Missing target labels",
        "Target column"
    ],
    "Value": [
        len(df),
        df.shape[1],
        int(df.duplicated().sum()),
        int(df["Class"].isna().sum()) if "Class" in df.columns else "Target not found",
        "Class"
    ]
})

display(overview)
display(df.head())

## Task / Question
**Apply EDA first: create a concise dataset overview, inspect feature types, summarize missing values, and report the fraud/legitimate class counts.**

In [ ]:

eda_overview = pd.DataFrame({
    "Column": df.columns,
    "Dtype": [str(t) for t in df.dtypes],
    "Missing values": df.isna().sum().values,
    "Unique values": [df[c].nunique(dropna=True) for c in df.columns]
})

class_counts = (
    df["Class"]
    .value_counts()
    .rename(index={0: "Legitimate", 1: "Fraud"})
    .rename_axis("Class")
    .reset_index(name="Count")
)
class_counts["Percentage"] = (100 * class_counts["Count"] / len(df)).round(4)

display(eda_overview.head(10))
display(class_counts)

## Task / Question
**Visualize the class imbalance clearly, because this is the central challenge of fraud detection.**

In [ ]:

plt.figure(figsize=(7, 4.5))
ax = sns.barplot(
    data=class_counts,
    x="Class",
    y="Count",
    palette=["#4C78A8", "#E45756"]
)
plt.title("Class distribution: legitimate vs fraud")
plt.ylabel("Number of transactions")
plt.xlabel("")
for p in ax.patches:
    ax.annotate(
        f"{int(p.get_height()):,}",
        (p.get_x() + p.get_width() / 2, p.get_height()),
        ha="center",
        va="bottom",
        fontsize=10
    )
plt.tight_layout()
plt.show()

imbalance_note = pd.DataFrame({
    "Observation": [
        "Fraud is a very small minority class.",
        "Accuracy alone would be misleading.",
        "Threshold choice matters operationally."
    ],
    "Implication": [
        "A naive model can look good by mostly predicting legitimate.",
        "Precision, recall, F1-score, balanced accuracy, and business cost should be emphasized.",
        "The final decision rule should reflect the business trade-off between missed fraud and customer friction."
    ]
})

display(imbalance_note)

## Task / Question
**Continue the EDA with one missing-value summary and two additional visualizations that reveal useful structure for later modelling.**

In [ ]:

missing_summary = (
    df.isna().sum()
    .rename("Missing values")
    .to_frame()
    .assign(Missing_pct=lambda x: 100 * x["Missing values"] / len(df))
    .sort_values("Missing values", ascending=False)
)

display(missing_summary.head(10))

fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))

sns.histplot(
    data=df.assign(LogAmount=np.log1p(df["Amount"])),
    x="LogAmount",
    hue="Class",
    bins=60,
    stat="density",
    common_norm=False,
    palette=["#4C78A8", "#E45756"],
    ax=axes[0]
)
axes[0].set_title("Log(Amount + 1) distribution by class")
axes[0].set_xlabel("log(Amount + 1)")
axes[0].legend(title="Class", labels=["Legitimate", "Fraud"])

time_bins = pd.qcut(df["Time"], q=20, duplicates="drop")
fraud_rate_by_time = (
    df.assign(TimeBin=time_bins)
      .groupby("TimeBin")["Class"]
      .mean()
      .reset_index(name="FraudRate")
)
fraud_rate_by_time["BinOrder"] = np.arange(len(fraud_rate_by_time))

sns.lineplot(
    data=fraud_rate_by_time,
    x="BinOrder",
    y="FraudRate",
    marker="o",
    color="#54A24B",
    ax=axes[1]
)
axes[1].set_title("Fraud rate across time-ordered bins")
axes[1].set_xlabel("Time bin (earliest to latest)")
axes[1].set_ylabel("Fraud rate")

plt.tight_layout()
plt.show()

## Task / Question
**State the binary target clearly and report the resulting class counts.**

In [ ]:

label_mapping = pd.DataFrame({
    "Original label": [0, 1],
    "Meaning": ["Legitimate transaction", "Fraudulent transaction"],
    "Binary target used": [0, 1]
})

display(label_mapping)
display(class_counts)

## Task / Question
**Provide short pseudocode for the time-aware split logic and explain why a time-aware evaluation is more realistic than a fully random split.**

```text
1. Sort the dataset by Time from earliest to latest.
2. Reserve the earliest 70% for training.
3. Reserve the next 15% for validation.
4. Reserve the latest 15% for testing.
5. Fit preprocessing and models only on the training split.
6. Use the validation split for model comparison and threshold selection.
7. Freeze the chosen model family and threshold.
8. Refit on train + validation if desired, then evaluate once on the latest test period.
```

**Why this is more realistic**

In real deployment, a fraud model is trained on **past** transactions and applied to **future** transactions.  
A random split mixes older and newer transactions together, which can leak future behavior patterns into training and make the evaluation look easier than the real operational setting.

## Task / Question
**Create a leakage-aware time-based split table and a simple split diagram. Also explain one realistic residual leakage risk.**

In [ ]:

df_time = df.sort_values("Time").reset_index(drop=True)

n = len(df_time)
train_end = int(0.70 * n)
val_end = int(0.85 * n)

train_df = df_time.iloc[:train_end].copy()
val_df = df_time.iloc[train_end:val_end].copy()
test_df = df_time.iloc[val_end:].copy()

split_table = pd.DataFrame([
    {
        "Split": "Train",
        "Rows": len(train_df),
        "Start Time": train_df["Time"].min(),
        "End Time": train_df["Time"].max(),
        "Fraud count": int(train_df["Class"].sum()),
        "Fraud rate": train_df["Class"].mean()
    },
    {
        "Split": "Validation",
        "Rows": len(val_df),
        "Start Time": val_df["Time"].min(),
        "End Time": val_df["Time"].max(),
        "Fraud count": int(val_df["Class"].sum()),
        "Fraud rate": val_df["Class"].mean()
    },
    {
        "Split": "Test",
        "Rows": len(test_df),
        "Start Time": test_df["Time"].min(),
        "End Time": test_df["Time"].max(),
        "Fraud count": int(test_df["Class"].sum()),
        "Fraud rate": test_df["Class"].mean()
    }
])

display(split_table.round(6))

plt.figure(figsize=(10, 2.6))
plt.hlines(y=1, xmin=train_df["Time"].min(), xmax=train_df["Time"].max(), color="#4C78A8", linewidth=10, label="Train")
plt.hlines(y=1, xmin=val_df["Time"].min(), xmax=val_df["Time"].max(), color="#F58518", linewidth=10, label="Validation")
plt.hlines(y=1, xmin=test_df["Time"].min(), xmax=test_df["Time"].max(), color="#E45756", linewidth=10, label="Test")
plt.yticks([])
plt.xlabel("Time")
plt.title("Time-aware split diagram")
plt.legend(loc="upper center", ncol=3, frameon=True)
plt.tight_layout()
plt.show()

leakage_audit = pd.DataFrame({
    "Item": [
        "Leakage prevented by design",
        "Residual leakage risk"
    ],
    "Explanation": [
        "Future transactions are not allowed to influence model fitting or threshold choice.",
        "Near-duplicate or highly related fraudulent campaigns can still appear across nearby time windows, so some similarity may remain even in a time-aware split."
    ]
})
display(leakage_audit)

## Task / Question
**Prepare the modelling features and add one short class-imbalance handling note.**

In [ ]:

FEATURES = [c for c in df_time.columns if c != "Class"]
TARGET = "Class"

X_train = train_df[FEATURES].copy()
y_train = train_df[TARGET].copy()

X_val = val_df[FEATURES].copy()
y_val = val_df[TARGET].copy()

X_test = test_df[FEATURES].copy()
y_test = test_df[TARGET].copy()

imbalance_handling_note = pd.DataFrame([
    [
        "Severe class imbalance",
        "Use class-weighted models so fraud errors receive more influence during training."
    ],
    [
        "Business cost asymmetry",
        "Tune the probability threshold on validation instead of relying on the default 0.50 threshold."
    ],
    [
        "Evaluation emphasis",
        "Report precision, recall, F1-score, balanced accuracy, and a business-style total cost."
    ]
], columns=["Issue", "Design choice"])

display(imbalance_handling_note)

## Task / Question
**Define one cost-sensitive decision rule and justify it in business terms.**

### Cost-sensitive rule
Use the following operational cost:

\[
\text{Total Cost} = 25 \times FN + 1 \times FP
\]

### Business justification
- A **false negative** means a fraudulent transaction is missed. This can cause direct financial loss, chargebacks, investigation effort, and reputational damage.
- A **false positive** means a legitimate customer may be delayed or reviewed, which still has a cost, but is usually much smaller than letting fraud pass through.

The ratio **25:1** is a simple business-oriented decision rule, not an exact currency model.  
It expresses the idea that **missed fraud is much more expensive than unnecessary review**.

## Task / Question
**Build one lighter baseline and one stronger model. Keep preprocessing visible and reproducible.**

In [ ]:

scale_features = ["Time", "Amount"]
pass_features = [c for c in FEATURES if c not in scale_features]

preprocessor = ColumnTransformer(
    transformers=[
        ("scale_time_amount", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", RobustScaler())
        ]), scale_features),
        ("pass_rest", "passthrough", pass_features)
    ]
)

baseline_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(
        max_iter=3000,
        class_weight="balanced",
        solver="liblinear",
        random_state=RANDOM_STATE
    ))
])

stronger_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=250,
        max_depth=None,
        min_samples_leaf=2,
        class_weight="balanced_subsample",
        n_jobs=-1,
        random_state=RANDOM_STATE
    ))
])

candidate_models = {
    "Logistic Regression (baseline)": baseline_model,
    "Random Forest (stronger)": stronger_model
}

## Task / Question
**Train both models on the training split and compare them on the validation split at the default threshold 0.50.**

In [ ]:

def evaluate_at_threshold(y_true, probas, threshold, fn_cost=25, fp_cost=1):
    preds = (probas >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, preds, labels=[0, 1]).ravel()
    return {
        "Threshold": threshold,
        "Confusion Matrix": f"TN={tn}, FP={fp}, FN={fn}, TP={tp}",
        "Precision": precision_score(y_true, preds, zero_division=0),
        "Recall": recall_score(y_true, preds, zero_division=0),
        "F1-score": f1_score(y_true, preds, zero_division=0),
        "Balanced Accuracy": balanced_accuracy_score(y_true, preds),
        "Total Cost": fn_cost * fn + fp_cost * fp
    }

model_rows = []
fitted_models = {}

for model_name, model in candidate_models.items():
    fitted = clone(model)
    fitted.fit(X_train, y_train)
    val_proba = fitted.predict_proba(X_val)[:, 1]

    row = evaluate_at_threshold(y_val, val_proba, threshold=0.50, fn_cost=25, fp_cost=1)
    row["Model"] = model_name
    model_rows.append(row)
    fitted_models[model_name] = fitted

model_comparison = pd.DataFrame(model_rows)[[
    "Model", "Threshold", "Confusion Matrix",
    "Precision", "Recall", "F1-score", "Balanced Accuracy", "Total Cost"
]].sort_values(by=["Total Cost", "Recall"], ascending=[True, False])

display(model_comparison.round(4))

best_model_name = model_comparison.iloc[0]["Model"]
best_model_train_only = fitted_models[best_model_name]
print(f"Chosen model family for threshold tuning: {best_model_name}")

## Task / Question
**Construct a threshold table for the chosen model using the validation split. Report at least five thresholds with total cost, confusion matrix, precision, recall, F1-score, and balanced accuracy.**

In [ ]:

val_proba_best = best_model_train_only.predict_proba(X_val)[:, 1]

thresholds = [0.05, 0.10, 0.20, 0.30, 0.50, 0.70]
threshold_rows = []

for threshold in thresholds:
    threshold_rows.append(
        evaluate_at_threshold(y_val, val_proba_best, threshold=threshold, fn_cost=25, fp_cost=1)
    )

threshold_table = pd.DataFrame(threshold_rows).sort_values(
    by=["Total Cost", "Recall", "F1-score"],
    ascending=[True, False, False]
).reset_index(drop=True)

display(threshold_table.round(4))

plt.figure(figsize=(8.5, 4.8))
sns.lineplot(
    data=threshold_table.sort_values("Threshold"),
    x="Threshold",
    y="Total Cost",
    marker="o",
    linewidth=2,
    color="#E45756"
)
plt.title("Validation total cost across candidate thresholds")
plt.ylabel("Total Cost = 25×FN + 1×FP")
plt.tight_layout()
plt.show()

chosen_threshold = float(threshold_table.iloc[0]["Threshold"])
print(f"Chosen validation threshold: {chosen_threshold:.2f}")

## Task / Question
**Refit the chosen model on Train + Validation, freeze the selected threshold, and evaluate once on the Test split.**

In [ ]:

train_val_df = pd.concat([train_df, val_df], axis=0).sort_values("Time")
X_train_val = train_val_df[FEATURES].copy()
y_train_val = train_val_df[TARGET].copy()

final_model = clone(candidate_models[best_model_name])
final_model.fit(X_train_val, y_train_val)

test_proba = final_model.predict_proba(X_test)[:, 1]
test_preds = (test_proba >= chosen_threshold).astype(int)

tn, fp, fn, tp = confusion_matrix(y_test, test_preds, labels=[0, 1]).ravel()

final_test_table = pd.DataFrame({
    "Metric": [
        "Threshold",
        "Confusion Matrix",
        "Precision",
        "Recall",
        "F1-score",
        "Balanced Accuracy",
        "Total Cost"
    ],
    "Value": [
        chosen_threshold,
        f"TN={tn}, FP={fp}, FN={fn}, TP={tp}",
        precision_score(y_test, test_preds, zero_division=0),
        recall_score(y_test, test_preds, zero_division=0),
        f1_score(y_test, test_preds, zero_division=0),
        balanced_accuracy_score(y_test, test_preds),
        25 * fn + fp
    ]
})

display(final_test_table.round(4))

## Task / Question
**Visualize the final confusion matrix on the test split.**

In [ ]:

cm = confusion_matrix(y_test, test_preds, labels=[0, 1])

plt.figure(figsize=(5.8, 4.8))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="YlGnBu",
    xticklabels=["Pred 0: Legitimate", "Pred 1: Fraud"],
    yticklabels=["True 0: Legitimate", "True 1: Fraud"]
)
plt.title("Final test confusion matrix")
plt.xlabel("Predicted class")
plt.ylabel("True class")
plt.tight_layout()
plt.show()

report_df = pd.DataFrame(
    classification_report(y_test, test_preds, output_dict=True)
).T

display(report_df.round(4))

## Task / Question
**Add one short deployment note explaining how the chosen threshold would affect analyst workload, customer friction, or missed-fraud risk.**

In [ ]:

alert_volume = int(test_preds.sum())
alert_rate = alert_volume / len(y_test)
missed_fraud = int(((y_test == 1) & (test_preds == 0)).sum())

deployment_note = pd.DataFrame([
    [
        "Analyst workload",
        f"The chosen threshold would send {alert_volume:,} of {len(y_test):,} test transactions for fraud action/review "
        f"({100*alert_rate:.3f}% of transactions). Lower thresholds increase workload."
    ],
    [
        "Customer friction",
        "Every extra false positive can delay or challenge a legitimate customer. This is why threshold selection cannot focus on recall alone."
    ],
    [
        "Missed-fraud risk",
        f"The current threshold misses {missed_fraud:,} fraudulent test transactions. If the business becomes more risk-averse, the threshold can be lowered to reduce FN at the cost of more alerts."
    ]
], columns=["Deployment aspect", "Reflection"])

display(deployment_note)

## Final reflection
This notebook uses a **time-aware** evaluation because fraud systems operate on future transactions, not on randomly mixed history.  
It also makes the **cost-sensitive logic visible** by turning threshold selection into a business trade-off between:
- catching more fraud
- creating more analyst workload
- increasing or reducing customer friction

That makes the final design more realistic than reporting a single default-threshold score in isolation.